In [5]:
# 01_lung_mechanics.ipynb – Enhanced Edition

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from IPython.display import display, clear_output

# ============================================================
# Core model
# ============================================================

def ventilator_ode(t, y, R, C, P_insp, PEEP, insp_time, rate):
    V = y[0]
    period = 60 / rate
    t_mod = t % period
    P_aw = P_insp if t_mod < insp_time else 0.0
    Q = (P_aw - V/C) / R
    if V <= 0 and Q < 0:
        Q = 0
    return [Q]

def simulate(R, C, P_insp, PEEP, rate, insp_frac=0.33, duration=10):
    insp_time = (60/rate) * insp_frac
    t_eval = np.linspace(0, duration, 2000)
    sol = solve_ivp(ventilator_ode, (0, duration), [0.0], t_eval=t_eval,
                    args=(R, C, P_insp, PEEP, insp_time, rate),
                    method='RK45', rtol=1e-6)
    t = sol.t
    V = sol.y[0]
    Q = np.gradient(V, t)
    P_aw = PEEP + V/C + R * Q
    return t, V, Q, P_aw

def compute_metrics(t, V, Q, P_aw, PEEP, rate):
    V_tidal = np.max(V) - np.min(V)
    minute_vent = V_tidal * rate
    peak_P = np.max(P_aw)
    mean_P = np.mean(P_aw)
    C_dyn = V_tidal / (peak_P - PEEP) if (peak_P - PEEP) > 0 else 0
    # Estimated resistance (simplified)
    insp_idx = np.where(P_aw > PEEP)[0]
    if len(insp_idx) > 5:
        Q_mid = np.median(Q[insp_idx])
        R_est = (peak_P - np.median(P_aw[insp_idx])) / Q_mid if Q_mid != 0 else 0
    else:
        R_est = 0
    return {
        'Vt_mL': V_tidal * 1000,
        'Minute_vent_Lmin': minute_vent,
        'Peak_P_cmH2O': peak_P,
        'Mean_P_cmH2O': mean_P,
        'C_dyn_mL_cmH2O': C_dyn * 1000,
        'R_est_cmH2O_Ls': R_est
    }

def plot_results(t, V, Q, P_aw, R, C, P_insp, PEEP, rate, insp_frac):
    metrics = compute_metrics(t, V, Q, P_aw, PEEP, rate)
    
    # Use constrained_layout instead of tight_layout
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
    ax1, ax2 = axes[0, 0], axes[0, 1]
    ax3, ax4 = axes[1, 0], axes[1, 1]
    
    # Pressure
    ax1.plot(t, P_aw, 'b-', lw=2)
    ax1.axhline(PEEP, color='gray', linestyle='--', label=f'PEEP = {PEEP} cmH2O')
    ax1.set_ylabel('Pressure (cmH2O)')
    ax1.set_title('Airway Pressure')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Flow
    ax2.plot(t, Q, 'g-', lw=2)
    ax2.axhline(0, color='k', lw=0.5)
    ax2.set_ylabel('Flow (L/s)')
    ax2.set_title('Flow Waveform')
    ax2.grid(True, alpha=0.3)
    
    # Volume
    ax3.plot(t, V*1000, 'r-', lw=2)
    ax3.set_xlabel('Time (s)')
    ax3.set_ylabel('Volume (mL)')
    ax3.set_title('Lung Volume')
    ax3.grid(True, alpha=0.3)
    ax3.annotate(f'Vt = {metrics["Vt_mL"]:.0f} mL', xy=(0.7, 0.9), xycoords='axes fraction',
                 bbox=dict(boxstyle="round", fc="white"))
    
    # Pressure‑Volume loop
    ax4.plot(P_aw, V*1000, 'purple', lw=1.5)
    ax4.scatter([P_aw[0]], [V[0]*1000], color='blue', s=30, label='Start inspiration')
    ax4.scatter([P_aw[np.argmax(V)]], [np.max(V)*1000], color='red', s=30, label='End inspiration')
    ax4.set_xlabel('Pressure (cmH2O)')
    ax4.set_ylabel('Volume (mL)')
    ax4.set_title('Pressure‑Volume Loop')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    fig.suptitle(f'Ventilator Simulation: R={R:.1f}, C={C:.3f}, Rate={rate} bpm, I:E=1:{1/insp_frac-1:.1f}',
                 fontsize=12)
    
    # Text box with metrics
    textstr = (f"Tidal Volume: {metrics['Vt_mL']:.0f} mL\n"
               f"Minute Ventilation: {metrics['Minute_vent_Lmin']:.2f} L/min\n"
               f"Peak Pressure: {metrics['Peak_P_cmH2O']:.1f} cmH2O\n"
               f"Mean Pressure: {metrics['Mean_P_cmH2O']:.1f} cmH2O\n"
               f"Dynamic Compliance: {metrics['C_dyn_mL_cmH2O']:.1f} mL/cmH2O\n"
               f"Estimated Resistance: {metrics['R_est_cmH2O_Ls']:.1f} cmH2O/L/s")
    
    ax1.text(0.02, 0.98, textstr, transform=ax1.transAxes, fontsize=9,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.show()
    
    # Console output
    print("\n📊 Clinical Metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.2f}")

# ============================================================
# Interactive widgets
# ============================================================

style = {'description_width': 'initial'}
R_slider = widgets.FloatSlider(value=8, min=2, max=30, step=0.5, description='Resistance (cmH2O/L/s):', style=style)
C_slider = widgets.FloatSlider(value=0.06, min=0.02, max=0.15, step=0.002, description='Compliance (L/cmH2O):', style=style)
P_insp_slider = widgets.FloatSlider(value=15, min=5, max=30, step=0.5, description='Inspiratory Pressure (cmH2O):', style=style)
PEEP_slider = widgets.FloatSlider(value=5, min=0, max=15, step=1, description='PEEP (cmH2O):', style=style)
rate_slider = widgets.IntSlider(value=15, min=8, max=30, step=1, description='Respiratory Rate (bpm):', style=style)
insp_frac_slider = widgets.FloatSlider(value=0.33, min=0.2, max=0.5, step=0.01, description='Inspiratory fraction (I:E ratio):', style=style)
duration_slider = widgets.FloatSlider(value=10, min=5, max=30, step=1, description='Simulation time (s):', style=style)

def update(R, C, P_insp, PEEP, rate, insp_frac, duration):
    clear_output(wait=True)
    t, V, Q, P_aw = simulate(R, C, P_insp, PEEP, rate, insp_frac, duration)
    plot_results(t, V, Q, P_aw, R, C, P_insp, PEEP, rate, insp_frac)

ui = widgets.VBox([
    widgets.HBox([R_slider, C_slider]),
    widgets.HBox([P_insp_slider, PEEP_slider]),
    widgets.HBox([rate_slider, insp_frac_slider]),
    duration_slider
])

out = widgets.interactive_output(update, {
    'R': R_slider, 'C': C_slider, 'P_insp': P_insp_slider, 'PEEP': PEEP_slider,
    'rate': rate_slider, 'insp_frac': insp_frac_slider, 'duration': duration_slider
})

display(ui, out)

Output()